# Prophet Experiment

Walmart Store Sales Forecasting — Prophet experiment

**MLflow Experiment:** `Prophet_Training`

---

In [1]:
import sys, os, warnings, json, tempfile, pickle, logging
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import prophet  # noqa: F401
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'prophet', '-q'], check=True)

from prophet import Prophet

# cmdstanpy logs an INFO line per chain per fit -- with ~280 individual Prophet fits in this
# notebook that's very noisy, so raise the threshold to WARNING.
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
logging.getLogger('prophet').setLevel(logging.WARNING)

import dagshub
dagshub.init(repo_owner='tgela23', repo_name='walmart-sales-forecasting', mlflow=True)
import mlflow
import mlflow.pyfunc
mlflow.set_experiment('Prophet_Training')

sys.path.insert(0, '/Users/r00t/Claude/Projects/ML final project')
from utils.feature_engineering import wmae, walk_forward_splits, MD_COLS

RANDOM_STATE = 42
TARGET = 'Weekly_Sales'
DATA_DIR = '/Users/r00t/Claude/Projects/ML final project/data/raw/walmart-recruiting-store-sales-forecasting/'
np.random.seed(RANDOM_STATE)

# Speed settings -- benchmarked at ~1.9s/fit, so 40 series x 3 folds x 2 seasonality configs
# (~240 fits) plus the 40-series Best_Model refit (~40 fits) totals ~9 minutes.
N_SERIES = 40
FORECAST_HORIZON = 8
N_CV_SPLITS = 3
REGRESSOR_COLS = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment'] + MD_COLS

# Prophet hyperparameters (shared across CV configs; only seasonality_mode is swept)
YEARLY_SEASONALITY = True
WEEKLY_SEASONALITY = False
N_CHANGEPOINTS = 25
CHANGEPOINT_PRIOR_SCALE = 0.05
SEASONALITY_PRIOR_SCALE = 10.0
HOLIDAYS_PRIOR_SCALE = 10.0

Importing plotly failed. Interactive plots will not work.


Accessing as tgela23

Initialized MLflow to track repo "tgela23/walmart-sales-forecasting"

Repository tgela23/walmart-sales-forecasting initialized!

2026/07/10 06:01:59 INFO mlflow.tracking.fluent: Experiment with name 'Prophet_Training' does not exist. Creating a new experiment.


## 1. Data Loading

In [2]:
train = pd.read_csv(DATA_DIR + 'train.csv', parse_dates=['Date'])
stores = pd.read_csv(DATA_DIR + 'stores.csv')
features = pd.read_csv(DATA_DIR + 'features.csv', parse_dates=['Date'])
test = pd.read_csv(DATA_DIR + 'test.csv', parse_dates=['Date'])

print(f'train: {train.shape} | stores: {stores.shape} | features: {features.shape} | test: {test.shape}')
print(f'Date range: {train["Date"].min().date()} -> {train["Date"].max().date()}')
print(f'n_stores={train["Store"].nunique()}, n_depts={train["Dept"].nunique()}')

train: (421570, 5) | stores: (45, 3) | features: (8190, 12) | test: (115064, 4)
Date range: 2010-02-05 -> 2012-10-26
n_stores=45, n_depts=81


## 2. Preprocessing & Feature Engineering

*MLflow run: `Prophet_Cleaning`*

In [3]:
with mlflow.start_run(run_name='Prophet_Cleaning') as run_cleaning:
    print('run_id:', run_cleaning.info.run_id)

    merged = train.merge(stores, on='Store', how='left')
    merged = merged.merge(features.drop(columns=['IsHoliday']), on=['Store', 'Date'], how='left')

    n_rows_raw = len(merged)

    markdown_null_pct_before_fill = float(merged[MD_COLS].isna().mean().mean() * 100)
    merged[MD_COLS] = merged[MD_COLS].fillna(0)

    n_negative_clipped = int((merged[TARGET] < 0).sum())
    merged[TARGET] = merged[TARGET].clip(lower=0)

    merged['IsHoliday'] = merged['IsHoliday'].astype(int)

    df_clean = merged.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
    n_rows_after_cleaning = len(df_clean)

    n_stores = int(df_clean['Store'].nunique())
    n_depts = int(df_clean['Dept'].nunique())
    holiday_week_pct = float(df_clean['IsHoliday'].mean() * 100)

    # Select top N_SERIES (Store, Dept) series by total Weekly_Sales -- keeps the 3-fold x
    # 2-config Prophet CV sweep fast (see the N_SERIES note in cell 1).
    top_series = df_clean.groupby(['Store', 'Dept'])[TARGET].sum().nlargest(N_SERIES).index
    df_selected = df_clean[df_clean.set_index(['Store', 'Dept']).index.isin(top_series)].reset_index(drop=True)
    n_series_selected = df_selected.groupby(['Store', 'Dept']).ngroups

    unique_dates = np.sort(df_selected['Date'].unique())
    val_dates = unique_dates[-FORECAST_HORIZON:]
    train_dates = unique_dates[:-FORECAST_HORIZON]
    val_holdout_start = str(pd.Timestamp(val_dates[0]).date())
    val_holdout_end = str(pd.Timestamp(val_dates[-1]).date())

    mlflow.log_param('n_rows_raw', n_rows_raw)
    mlflow.log_param('n_rows_after_cleaning', n_rows_after_cleaning)
    mlflow.log_metric('n_negative_clipped', n_negative_clipped)
    mlflow.log_metric('markdown_null_pct_before_fill', markdown_null_pct_before_fill)
    mlflow.log_param('n_stores', n_stores)
    mlflow.log_param('n_depts', n_depts)
    mlflow.log_param('n_series_selected', n_series_selected)
    mlflow.log_param('val_holdout_start', val_holdout_start)
    mlflow.log_param('val_holdout_end', val_holdout_end)
    mlflow.log_metric('holiday_week_pct', holiday_week_pct)

    print(f'Raw merged rows: {n_rows_raw} -> cleaned: {n_rows_after_cleaning}')
    print(f'Negative sales clipped: {n_negative_clipped} | MarkDown null% before fill: {markdown_null_pct_before_fill:.1f}%')
    print(f'n_stores={n_stores}, n_depts={n_depts}, n_series_selected={n_series_selected}')
    print(f'Holiday week %: {holiday_week_pct:.2f}% | Holdout: {val_holdout_start} -> {val_holdout_end}')

df_selected.head()

run_id: ba4d937d072746d69b10db288594af23


Raw merged rows: 421570 -> cleaned: 421570
Negative sales clipped: 1285 | MarkDown null% before fill: 67.5%
n_stores=45, n_depts=81, n_series_selected=40
Holiday week %: 7.04% | Holdout: 2012-09-07 -> 2012-10-26


🏃 View run Prophet_Cleaning at: https://dagshub.com/tgela23/walmart-sales-forecasting.mlflow/#/experiments/5/runs/ba4d937d072746d69b10db288594af23
🧪 View experiment at: https://dagshub.com/tgela23/walmart-sales-forecasting.mlflow/#/experiments/5


,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment
0,1,92,2010-02-05,139884.94,0,A,151315,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106
1,1,92,2010-02-12,143081.42,1,A,151315,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106
2,1,92,2010-02-19,135066.75,0,A,151315,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106
3,1,92,2010-02-26,125048.08,0,A,151315,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106
4,1,92,2010-03-05,132945.44,0,A,151315,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106


## 3. Feature Selection

*MLflow run: `Prophet_Feature_Selection`*

In [4]:
with mlflow.start_run(run_name='Prophet_Feature_Selection') as run_fs:
    print('run_id:', run_fs.info.run_id)

    # Prophet doesn't do feature/column selection the way tree models do -- this run instead
    # defines and logs the model configuration: seasonality settings, the Walmart holiday
    # calendar (as a Prophet `holidays` DataFrame), and the standardized external regressors.
    mlflow.log_param('seasonality_mode', 'multiplicative')
    mlflow.log_param('yearly_seasonality', YEARLY_SEASONALITY)
    mlflow.log_param('weekly_seasonality', WEEKLY_SEASONALITY)
    mlflow.log_param('n_changepoints', N_CHANGEPOINTS)
    mlflow.log_param('changepoint_prior_scale', CHANGEPOINT_PRIOR_SCALE)
    mlflow.log_param('seasonality_prior_scale', SEASONALITY_PRIOR_SCALE)
    mlflow.log_param('holidays_prior_scale', HOLIDAYS_PRIOR_SCALE)

    # Official Kaggle competition holiday weeks (2010-2013) -- lower/upper_window=0 since
    # these are already the specific marked weeks, not date ranges to expand around.
    HOLIDAY_DATES = {
        'Super_Bowl': ['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08'],
        'Labor_Day': ['2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06'],
        'Thanksgiving': ['2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29'],
        'Christmas': ['2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27'],
    }
    holidays_df = pd.DataFrame([
        {'holiday': name, 'ds': pd.Timestamp(d), 'lower_window': 0, 'upper_window': 0}
        for name, dates in HOLIDAY_DATES.items() for d in dates
    ])
    n_holiday_events = len(holidays_df)

    # Standardize regressors using the selected series' own mean/std (avoids leaking val-fold
    # statistics differently than train-fold ones since this scaling is applied uniformly).
    reg_stats = {
        col: (float(df_selected[col].mean()), float(df_selected[col].std()) or 1.0)
        for col in REGRESSOR_COLS
    }
    n_regressors = len(REGRESSOR_COLS)

    mlflow.log_param('n_regressors', n_regressors)
    mlflow.log_param('n_holiday_events', n_holiday_events)
    mlflow.log_param('n_series_selected', n_series_selected)
    mlflow.log_dict({'regressors': REGRESSOR_COLS, 'holidays': list(HOLIDAY_DATES.keys())}, 'feature_config.json')

    print(f'Regressors ({n_regressors}): {REGRESSOR_COLS}')
    print(f'Holiday events ({n_holiday_events}) across {len(HOLIDAY_DATES)} holiday types: {list(HOLIDAY_DATES.keys())}')
    print(holidays_df)

run_id: c18d50f2720648e78d6f4c7ed50cf05f


Regressors (9): ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
Holiday events (16) across 4 holiday types: ['Super_Bowl', 'Labor_Day', 'Thanksgiving', 'Christmas']
         holiday         ds  lower_window  upper_window
0     Super_Bowl 2010-02-12             0             0
1     Super_Bowl 2011-02-11             0             0
2     Super_Bowl 2012-02-10             0             0
3     Super_Bowl 2013-02-08             0             0
4      Labor_Day 2010-09-10             0             0
5      Labor_Day 2011-09-09             0             0
6      Labor_Day 2012-09-07             0             0
7      Labor_Day 2013-09-06             0             0
8   Thanksgiving 2010-11-26             0             0
9   Thanksgiving 2011-11-25             0             0
10  Thanksgiving 2012-11-23             0             0
11  Thanksgiving 2013-11-29             0             0
12     Christmas 2010-12-31          

🏃 View run Prophet_Feature_Selection at: https://dagshub.com/tgela23/walmart-sales-forecasting.mlflow/#/experiments/5/runs/c18d50f2720648e78d6f4c7ed50cf05f
🧪 View experiment at: https://dagshub.com/tgela23/walmart-sales-forecasting.mlflow/#/experiments/5


## 4. Model Training & Cross-Validation

*MLflow run: `Prophet_CV`*

In [5]:
sampled_pairs = list(df_selected.groupby(['Store', 'Dept']).groups.keys())
cv_splits = walk_forward_splits(unique_dates, n_splits=N_CV_SPLITS, val_size=FORECAST_HORIZON)


def make_prophet(seasonality_mode):
    m = Prophet(
        seasonality_mode=seasonality_mode,
        yearly_seasonality=YEARLY_SEASONALITY,
        weekly_seasonality=WEEKLY_SEASONALITY,
        n_changepoints=N_CHANGEPOINTS,
        changepoint_prior_scale=CHANGEPOINT_PRIOR_SCALE,
        seasonality_prior_scale=SEASONALITY_PRIOR_SCALE,
        holidays_prior_scale=HOLIDAYS_PRIOR_SCALE,
        holidays=holidays_df,
    )
    for col in REGRESSOR_COLS:
        m.add_regressor(col)
    return m


def prep_prophet_df(series_df):
    d = series_df[['Date', TARGET] + REGRESSOR_COLS].rename(columns={'Date': 'ds', TARGET: 'y'}).copy()
    for col in REGRESSOR_COLS:
        mean_, std_ = reg_stats[col]
        d[col] = (d[col] - mean_) / std_
    return d


def fit_predict_fold(series_df, train_cutoff_date, val_dates, seasonality_mode):
    d = prep_prophet_df(series_df)
    train_d = d[d['ds'] <= train_cutoff_date]
    future_d = d[d['ds'].isin(val_dates)]

    m = make_prophet(seasonality_mode)
    m.fit(train_d)
    forecast = m.predict(future_d[['ds'] + REGRESSOR_COLS])
    preds = np.clip(forecast['yhat'].values, 0, None)
    return preds, m, forecast


def evaluate_config(seasonality_mode):
    """Fits/evaluates all sampled series across all CV folds for one seasonality_mode.

    Returns a (n_series, n_folds) WMAE matrix, the count of series that failed to fit at
    least once, and the fitted model/forecast from the last fold's first series (for the
    sample plots).
    """
    scores = np.full((len(sampled_pairs), len(cv_splits)), np.nan)
    failed_series = set()
    sample_model, sample_forecast, sample_key = None, None, None

    for fold_i, (tr_dates, va_dates) in enumerate(cv_splits):
        for s_i, (store, dept) in enumerate(sampled_pairs):
            series_df = df_selected[(df_selected['Store'] == store) & (df_selected['Dept'] == dept)].sort_values('Date')
            try:
                preds, m, forecast = fit_predict_fold(series_df, tr_dates[-1], va_dates, seasonality_mode)
                val_part = series_df[series_df['Date'].isin(va_dates)]
                scores[s_i, fold_i] = wmae(val_part[TARGET].values, preds, val_part['IsHoliday'].values)
                if fold_i == len(cv_splits) - 1 and sample_model is None:
                    sample_model, sample_forecast, sample_key = m, forecast, (store, dept)
            except Exception:
                failed_series.add((store, dept))

    return scores, len(failed_series), sample_model, sample_forecast, sample_key


with mlflow.start_run(run_name='Prophet_CV') as run_cv:
    print('run_id:', run_cv.info.run_id)

    mlflow.log_param('changepoint_prior_scale', CHANGEPOINT_PRIOR_SCALE)
    mlflow.log_param('seasonality_prior_scale', SEASONALITY_PRIOR_SCALE)
    mlflow.log_param('holidays_prior_scale', HOLIDAYS_PRIOR_SCALE)
    mlflow.log_param('n_changepoints', N_CHANGEPOINTS)
    mlflow.log_param('n_regressors', len(REGRESSOR_COLS))
    mlflow.log_param('n_series', len(sampled_pairs))
    mlflow.log_param('n_cv_splits', len(cv_splits))

    config_results = {}
    for seasonality_mode in ['multiplicative', 'additive']:
        print(f'\n--- Evaluating seasonality_mode={seasonality_mode} ---')
        scores, n_failed, sample_model, sample_forecast, sample_key = evaluate_config(seasonality_mode)
        mean_score = float(np.nanmean(scores))
        config_results[seasonality_mode] = {
            'scores': scores, 'n_failed': n_failed, 'mean': mean_score,
            'sample_model': sample_model, 'sample_forecast': sample_forecast, 'sample_key': sample_key,
        }
        print(f'{seasonality_mode}: mean_cv_wmae={mean_score:.2f} (n_failed_series={n_failed})')

    best_seasonality_mode = min(config_results, key=lambda k: config_results[k]['mean'])
    best = config_results[best_seasonality_mode]
    scores = best['scores']

    per_fold_wmae = np.nanmean(scores, axis=0)
    for i, score in enumerate(per_fold_wmae):
        mlflow.log_metric(f'wmae_fold_{i}', float(score))

    mean_cv_wmae = float(np.nanmean(scores))
    std_cv_wmae = float(np.nanstd(scores))
    median_cv_wmae = float(np.nanmedian(scores))
    min_cv_wmae = float(np.nanmin(scores))
    max_cv_wmae = float(np.nanmax(scores))
    per_series_mean = np.nanmean(scores, axis=1)
    pct_series_under_2000_wmae = float(np.mean(per_series_mean < 2000) * 100)

    mlflow.log_metric('mean_cv_wmae', mean_cv_wmae)
    mlflow.log_metric('std_cv_wmae', std_cv_wmae)
    mlflow.log_metric('median_cv_wmae', median_cv_wmae)
    mlflow.log_metric('min_cv_wmae', min_cv_wmae)
    mlflow.log_metric('max_cv_wmae', max_cv_wmae)
    mlflow.log_metric('pct_series_under_2000_wmae', pct_series_under_2000_wmae)
    mlflow.log_metric('n_series_failed', best['n_failed'])
    mlflow.log_param('best_seasonality_mode', best_seasonality_mode)
    mlflow.log_metric('best_mean_cv_wmae', best['mean'])

    print(f'\nBest config: seasonality_mode={best_seasonality_mode}')
    print(f'CV WMAE -- mean: {mean_cv_wmae:.2f}, std: {std_cv_wmae:.2f}, median: {median_cv_wmae:.2f} '
          f'(min: {min_cv_wmae:.2f}, max: {max_cv_wmae:.2f})')
    print(f'% series with mean WMAE < 2000: {pct_series_under_2000_wmae:.1f}% | failed series: {best["n_failed"]}')

    # --- cv_wmae_distribution.png ---
    fig, ax = plt.subplots(figsize=(10, 5))
    sorted_scores = np.sort(per_series_mean[~np.isnan(per_series_mean)])
    ax.bar(range(len(sorted_scores)), sorted_scores, color=sns.color_palette('deep')[0])
    ax.axhline(mean_cv_wmae, color='black', linestyle='--', label=f'mean = {mean_cv_wmae:.0f}')
    ax.set_xlabel('Series (sorted)')
    ax.set_ylabel('Mean WMAE across folds')
    ax.set_title(f'Per-series CV WMAE distribution ({best_seasonality_mode})')
    ax.legend()
    plt.tight_layout()

    dist_path = os.path.join(tempfile.gettempdir(), 'cv_wmae_distribution.png')
    fig.savefig(dist_path, dpi=100, bbox_inches='tight')
    plt.close(fig)
    mlflow.log_artifact(dist_path)
    os.remove(dist_path)

    # --- sample_forecast_plot.png / components_plot.png (best config, last fold, first series) ---
    sample_model, sample_forecast, sample_key = best['sample_model'], best['sample_forecast'], best['sample_key']

    fig1 = sample_model.plot(sample_forecast)
    fig1.gca().set_title(f'Sample forecast -- Store {sample_key[0]} Dept {sample_key[1]} ({best_seasonality_mode})')
    forecast_path = os.path.join(tempfile.gettempdir(), 'sample_forecast_plot.png')
    fig1.savefig(forecast_path, dpi=100, bbox_inches='tight')
    plt.close(fig1)
    mlflow.log_artifact(forecast_path)
    os.remove(forecast_path)

    fig2 = sample_model.plot_components(sample_forecast)
    components_path = os.path.join(tempfile.gettempdir(), 'components_plot.png')
    fig2.savefig(components_path, dpi=100, bbox_inches='tight')
    plt.close(fig2)
    mlflow.log_artifact(components_path)
    os.remove(components_path)

run_id: f68a41be059f47c1b580b42e89cd34e8


06:02:31 - cmdstanpy - INFO - Chain [1] start processing



--- Evaluating seasonality_mode=multiplicative ---


06:02:31 - cmdstanpy - INFO - Chain [1] done processing


06:02:32 - cmdstanpy - INFO - Chain [1] start processing


...

06:03:15 - cmdstanpy - INFO - Chain [1] start processing


06:03:15 - cmdstanpy - INFO - Chain [1] done processing


additive: mean_cv_wmae=8215.09 (n_failed_series=0)



Best config: seasonality_mode=additive
CV WMAE -- mean: 8215.09, std: 5540.96, median: 6409.15 (min: 2031.77, max: 33309.75)
% series with mean WMAE < 2000: 0.0% | failed series: 0


🏃 View run Prophet_CV at: https://dagshub.com/tgela23/walmart-sales-forecasting.mlflow/#/experiments/5/runs/f68a41be059f47c1b580b42e89cd34e8
🧪 View experiment at: https://dagshub.com/tgela23/walmart-sales-forecasting.mlflow/#/experiments/5


## 5. Best Model — Save as Pipeline

*Saved to MLflow Model Registry*

In [6]:
class ProphetForecaster(mlflow.pyfunc.PythonModel):
    """Wraps the per-series fitted Prophet models + regressor standardization stats.

    Prophet models pickle cleanly at ~20-25KB each (unlike ARIMA/TFT, no lightweight
    reconstruction trick needed), so the whole dict is stored directly.
    """

    def __init__(self, models, reg_stats, regressor_cols):
        self.models = models
        self.reg_stats = reg_stats
        self.regressor_cols = regressor_cols

    def predict(self, context, model_input):
        # model_input: DataFrame with Store, Dept, Date, and regressor_cols for the requested dates
        results = []
        for (store, dept), grp in model_input.groupby(['Store', 'Dept']):
            grp = grp.sort_values('Date')
            m = self.models.get((store, dept))
            if m is None:
                results.append({'Store': store, 'Dept': dept, 'forecast': [float('nan')] * len(grp)})
                continue
            d = grp[['Date'] + self.regressor_cols].rename(columns={'Date': 'ds'}).copy()
            for col in self.regressor_cols:
                mean_, std_ = self.reg_stats[col]
                d[col] = (d[col] - mean_) / std_
            forecast = m.predict(d)
            preds = np.clip(forecast['yhat'].values, 0, None)
            results.append({'Store': store, 'Dept': dept, 'forecast': preds.tolist()})
        return pd.DataFrame(results)


with mlflow.start_run(run_name='Prophet_Best_Model') as run_best:
    print('run_id:', run_best.info.run_id)

    mlflow.log_param('seasonality_mode', best_seasonality_mode)
    mlflow.log_param('yearly_seasonality', YEARLY_SEASONALITY)
    mlflow.log_param('weekly_seasonality', WEEKLY_SEASONALITY)
    mlflow.log_param('n_changepoints', N_CHANGEPOINTS)
    mlflow.log_param('changepoint_prior_scale', CHANGEPOINT_PRIOR_SCALE)
    mlflow.log_param('seasonality_prior_scale', SEASONALITY_PRIOR_SCALE)
    mlflow.log_param('holidays_prior_scale', HOLIDAYS_PRIOR_SCALE)
    mlflow.log_param('n_regressors', len(REGRESSOR_COLS))

    fitted_models = {}
    orders_summary = {}
    train_scores, holdout_scores = [], []
    n_failed = 0

    for store, dept in sampled_pairs:
        series_df = df_selected[(df_selected['Store'] == store) & (df_selected['Dept'] == dept)].sort_values('Date')
        d = prep_prophet_df(series_df)
        train_d = d[d['ds'].isin(train_dates)]
        holdout_d = d[d['ds'].isin(val_dates)]

        try:
            m = make_prophet(best_seasonality_mode)
            m.fit(train_d)

            # In-sample sanity check only (not a generalization estimate)
            in_sample_fc = m.predict(train_d[['ds'] + REGRESSOR_COLS])
            in_sample_preds = np.clip(in_sample_fc['yhat'].values, 0, None)
            train_part = series_df[series_df['Date'].isin(train_dates)]
            train_scores.append(wmae(train_part[TARGET].values, in_sample_preds, train_part['IsHoliday'].values))

            # True holdout evaluation (last FORECAST_HORIZON weeks, never touched during CV)
            holdout_fc = m.predict(holdout_d[['ds'] + REGRESSOR_COLS])
            holdout_preds = np.clip(holdout_fc['yhat'].values, 0, None)
            holdout_part = series_df[series_df['Date'].isin(val_dates)]
            holdout_scores.append(wmae(holdout_part[TARGET].values, holdout_preds, holdout_part['IsHoliday'].values))

            fitted_models[(store, dept)] = m
            orders_summary[f'{store}_{dept}'] = {
                'seasonality_mode': best_seasonality_mode,
                'n_changepoints': N_CHANGEPOINTS,
                'n_train_obs': int(len(train_d)),
            }
        except Exception:
            n_failed += 1

    n_series_fitted = len(fitted_models)
    final_train_wmae = float(np.mean(train_scores)) if train_scores else float('nan')
    val_holdout_wmae = float(np.mean(holdout_scores)) if holdout_scores else float('nan')

    mlflow.log_param('n_series_fitted', n_series_fitted)
    mlflow.log_metric('n_failed', n_failed)
    mlflow.log_metric('final_train_wmae', final_train_wmae)
    mlflow.log_metric('val_holdout_wmae', val_holdout_wmae)

    print(f'Fitted {n_series_fitted}/{len(sampled_pairs)} series ({n_failed} failed)')
    print(f'Final in-sample WMAE (sanity check only): {final_train_wmae:.2f}')
    print(f'Holdout WMAE: {val_holdout_wmae:.2f}')
    print(f'For comparison -- walk-forward CV WMAE: {mean_cv_wmae:.2f}')

    orders_json_path = os.path.join(tempfile.gettempdir(), 'orders_summary.json')
    with open(orders_json_path, 'w') as f:
        json.dump(orders_summary, f, indent=2)
    mlflow.log_artifact(orders_json_path)
    os.remove(orders_json_path)

    forecaster = ProphetForecaster(models=fitted_models, reg_stats=reg_stats, regressor_cols=REGRESSOR_COLS)
    input_example = df_selected[df_selected.set_index(['Store', 'Dept']).index.isin(list(fitted_models.keys())[:3])]

    mlflow.pyfunc.log_model(
        name='Prophet_pipeline',
        python_model=forecaster,
        registered_model_name='Prophet_Pipeline',
        input_example=input_example,
    )

    print(f'\nModel registered as Prophet_Pipeline (run_id={run_best.info.run_id})')

    # --- Sanity check: reload from the registry and predict on 3 sample series ---
    client = mlflow.MlflowClient()
    versions = client.search_model_versions("name='Prophet_Pipeline'")
    latest_version = max(int(v.version) for v in versions)

    loaded_model = mlflow.pyfunc.load_model(f'models:/Prophet_Pipeline/{latest_version}')
    sample_preds = loaded_model.predict(input_example)
    print(f'\nLoaded Prophet_Pipeline version {latest_version}')
    print(sample_preds)

    print('\n=== Summary ===')
    print(f'n_series_fitted   : {n_series_fitted}')
    print(f'best_seasonality  : {best_seasonality_mode}')
    print(f'mean_cv_wmae      : {mean_cv_wmae:.2f}')
    print(f'val_holdout_wmae  : {val_holdout_wmae:.2f}')
    print(f'final_train_wmae : {final_train_wmae:.2f}')

/Users/r00t/Claude/Projects/ML final project/.venv/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


run_id: e3b27cd8e421411dad22cd0d1b3cf599


06:03:24 - cmdstanpy - INFO - Chain [1] start processing


06:03:24 - cmdstanpy - INFO - Chain [1] done processing


06:03:24 - cmdstanpy - INFO - Chain [1] start processing


...

2026/07/10 06:04:11 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Prophet_Pipeline, version 1


Created version '1' of model 'Prophet_Pipeline'.



Model registered as Prophet_Pipeline (run_id=e3b27cd8e421411dad22cd0d1b3cf599)



Loaded Prophet_Pipeline version 1
   Store  Dept                                           forecast
0      1    92  [136852.2882277246, 148771.18496681403, 134542...
1      1    95  [103721.72185333031, 114186.4761300197, 105699...
2      2    90  [106700.80175633717, 107864.67145948304, 99573...

=== Summary ===
n_series_fitted   : 40
best_seasonality  : additive
mean_cv_wmae      : 8215.09
val_holdout_wmae  : 6932.91
final_train_wmae : 5347.06


🏃 View run Prophet_Best_Model at: https://dagshub.com/tgela23/walmart-sales-forecasting.mlflow/#/experiments/5/runs/e3b27cd8e421411dad22cd0d1b3cf599
🧪 View experiment at: https://dagshub.com/tgela23/walmart-sales-forecasting.mlflow/#/experiments/5
